In [69]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize
import torch
import typing as t
from torch.utils.data import TensorDataset, DataLoader

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier # For baseline model
from sklearn.feature_extraction.text import TfidfVectorizer # To convert text to numbers
from sklearn.linear_model import LogisticRegression # The classifier model
from sklearn.metrics import accuracy_score, classification_report # For evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # For splitting and validating
from sklearn.pipeline import Pipeline # To chain processing steps
from matplotlib import pyplot as plt

In [70]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [133]:
# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('../Data/train.jsonl', lines=True)

# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('../Data/kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train_raw = train_data.drop('label', axis=1)
y_train_raw = train_data['label']

X_kaggle_raw = kaggle_data

In [183]:
# Define a function to get the full text from a tweet object.
# Tweets can be truncated, storing the full version in 'extended_tweet.full_text'.
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

# Apply this function to every row (axis=1) in the training data
X_train = X_train_raw.copy()
X_train['full_text'] = X_train_raw.apply(lambda tweet: extract_full_text(tweet), axis=1)
# Apply the same function to the Kaggle test data
X_kaggle = X_kaggle_raw.copy()
X_kaggle['full_text'] = X_kaggle_raw.apply(lambda tweet: extract_full_text(tweet), axis=1)

# Data Analysis

In [209]:
def clean_df(df):
    df.drop(columns=[
        'retweet_count', 'retweeted', 'favorite_count', 'favorited', 'quote_count', 'lang', 'geo',
        'in_reply_to_screen_name', 'in_reply_to_status_id', 'in_reply_to_user_id', 'in_reply_to_status_id_str',
        'filter_level', 'id_str', 'coordinates', 'quoted_status_id_str', 'contributors', 'extended_entities',
        'in_reply_to_user_id_str', 'quoted_status_id', 'withheld_in_countries', 'place', 'reply_count',
        'challenge_id', 'text', 'truncated', 'extended_tweet', 'display_text_range'], inplace=True, errors='ignore')
    return df

In [210]:
X_train = clean_df(X_train)
X_kaggle = clean_df(X_kaggle)

In [211]:
X_train['extended_tweet'][4]

KeyError: 'extended_tweet'

In [212]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
X_train.head(20)

,created_at,source,is_quote_status,timestamp_ms,possibly_sensitive,quoted_status.extended_tweet.entities.urls,quoted_status.extended_tweet.entities.hashtags,quoted_status.extended_tweet.entities.user_mentions,quoted_status.extended_tweet.entities.symbols,quoted_status.extended_tweet.full_text,quoted_status.extended_tweet.display_text_range,quoted_status.in_reply_to_status_id_str,quoted_status.in_reply_to_status_id,quoted_status.created_at,quoted_status.in_reply_to_user_id_str,quoted_status.source,quoted_status.retweet_count,quoted_status.retweeted,quoted_status.geo,quoted_status.filter_level,quoted_status.in_reply_to_screen_name,quoted_status.is_quote_status,quoted_status.id_str,quoted_status.in_reply_to_user_id,quoted_status.favorite_count,quoted_status.id,quoted_status.text,quoted_status.place,quoted_status.lang,quoted_status.quote_count,quoted_status.favorited,quoted_status.coordinates,quoted_status.truncated,quoted_status.reply_count,quoted_status.entities.urls,quoted_status.entities.hashtags,quoted_status.entities.user_mentions,quoted_status.entities.symbols,quoted_status.contributors,quoted_status.user.utc_offset,quoted_status.user.friends_count,quoted_status.user.profile_image_url_https,quoted_status.user.listed_count,quoted_status.user.profile_background_image_url,quoted_status.user.default_profile_image,quoted_status.user.favourites_count,quoted_status.user.description,quoted_status.user.created_at,quoted_status.user.is_translator,quoted_status.user.profile_background_image_url_https,quoted_status.user.protected,quoted_status.user.screen_name,quoted_status.user.id_str,quoted_status.user.profile_link_color,quoted_status.user.translator_type,quoted_status.user.id,quoted_status.user.geo_enabled,quoted_status.user.profile_background_color,quoted_status.user.lang,quoted_status.user.profile_sidebar_border_color,quoted_status.user.profile_text_color,quoted_status.user.verified,quoted_status.user.profile_image_url,quoted_status.user.time_zone,quoted_status.user.url,quoted_status.user.contributors_enabled,quoted_status.user.profile_background_tile,quoted_status.user.profile_banner_url,quoted_status.user.statuses_count,quoted_status.user.follow_request_sent,quoted_status.user.followers_count,quoted_status.user.profile_use_background_image,quoted_status.user.default_profile,quoted_status.user.following,quoted_status.user.name,quoted_status.user.location,quoted_status.user.profile_sidebar_fill_color,quoted_status.user.notifications,quoted_status_permalink.expanded,quoted_status_permalink.display,quoted_status_permalink.url,entities.urls,entities.hashtags,entities.user_mentions,entities.symbols,user.utc_offset,user.profile_image_url_https,user.listed_count,user.profile_background_image_url,user.default_profile_image,user.favourites_count,user.description,user.created_at,user.is_translator,user.profile_background_image_url_https,user.protected,user.profile_link_color,user.translator_type,user.geo_enabled,user.profile_background_color,user.lang,user.profile_sidebar_border_color,user.profile_text_color,user.profile_image_url,user.time_zone,user.url,user.contributors_enabled,user.profile_background_tile,user.profile_banner_url,user.statuses_count,user.follow_request_sent,user.profile_use_background_image,user.default_profile,user.following,user.location,user.profile_sidebar_fill_color,user.notifications,quoted_status,quoted_status_permalink,extended_tweet.entities.urls,extended_tweet.entities.hashtags,extended_tweet.entities.user_mentions,extended_tweet.entities.symbols,extended_tweet.full_text,extended_tweet.display_text_range,quoted_status.possibly_sensitive,quoted_status.extended_entities.media,quoted_status.entities.media,quoted_status.display_text_range,extended_tweet.extended_entities.media,extended_tweet.entities.media,quoted_status.extended_tweet.extended_entities.media,quoted_status.extended_tweet.entities.media,place.country_code,place.country,place.full_name,place.bounding_box.coordinates,place.bounding_box.type,place.place

In [79]:
list(X_train.columns)

['in_reply_to_status_id_str',
 'in_reply_to_status_id',
 'created_at',
 'in_reply_to_user_id_str',
 'source',
 'quoted_status_id',
 'retweet_count',
 'retweeted',
 'geo',
 'filter_level',
 'in_reply_to_screen_name',
 'is_quote_status',
 'id_str',
 'in_reply_to_user_id',
 'favorite_count',
 'text',
 'place',
 'lang',
 'quote_count',
 'favorited',
 'coordinates',
 'truncated',
 'timestamp_ms',
 'reply_count',
 'quoted_status_id_str',
 'contributors',
 'challenge_id',
 'extended_tweet',
 'display_text_range',
 'possibly_sensitive',
 'extended_entities',
 'withheld_in_countries',
 'quoted_status.extended_tweet.entities.urls',
 'quoted_status.extended_tweet.entities.hashtags',
 'quoted_status.extended_tweet.entities.user_mentions',
 'quoted_status.extended_tweet.entities.symbols',
 'quoted_status.extended_tweet.full_text',
 'quoted_status.extended_tweet.display_text_range',
 'quoted_status.in_reply_to_status_id_str',
 'quoted_status.in_reply_to_status_id',
 'quoted_status.created_at',
 'quo

In [ ]:
for idx in ['favorite_count', 'quote_count', 'favorited', 'reply_count', 'quoted_status_id', 'possibly_sensitive', 'retweet_count', 'retweeted']:
    print(idx, X_train[idx].unique())

is_quote_status [ True False]
favorite_count [0]
quote_count [0]
favorited [False]
reply_count [0]
quoted_status_id [1.37217083e+18            nan 1.37218184e+18 ... 1.37687645e+18
 1.37698388e+18 1.37693953e+18]
possibly_sensitive [nan  0.  1.]
retweet_count [0]
retweeted [False]


In [48]:
def simplify_source(src):
    if src is None:
        return "unknown"
    if str(src) == 'nan':
        return "unknown"
    s = src.lower()

    if "twitter" in s:
        return "twitter_official"
    if any(x in s for x in ["buffer", "hootsuite", "publer", "sprinklr", "tweetdeck", "ifttt", "post", "planable"]):
        return "scheduler"
    if any(x in s for x in ["bot", "auto", "rss", "feed", "revive", "publisher"]):
        return "automation"
    if any(x in s for x in ["instagram", "insta"]):
        return "instagram"
    if any(x in s for x in ["linkedin"]):
        return "linkedin"
    if any(x in s for x in ["press", "journal", "news", "mag", "africa", "france"]):
        return "media"
    
    return "other"

In [52]:
def extract_features(df: pd.DataFrame) -> pd.DataFrame:
    features = pd.DataFrame()

    # Viralité
    features["log_retweets"] = np.log1p(df["retweet_count"])
    features["log_favorites"] = np.log1p(df["favorite_count"])
    features["log_replies"] = np.log1p(df["reply_count"])
    features["log_quotes"] = np.log1p(df["quote_count"])

    # Engagement structurel
    features["is_reply"] = df["in_reply_to_status_id"].notnull().astype(int)
    features["is_quote"] = df["quoted_status_id"].notnull().astype(int)

    # Source du tweet
    features["source_clean"] = df["source"].str.extract(r'>(.*?)<')
    features["source_group"] = features["source_clean"].apply(simplify_source)
    source_dummies = pd.get_dummies(features, columns=["source_group"]).drop(columns=["source_clean"])

    # Date features
    features["hour"] = df["created_at"].dt.hour
    features["dayofweek"] = df["created_at"].dt.dayofweek
    return pd.concat([features.drop(columns=["source_clean"]), source_dummies], axis=1)

In [ ]:
features_train = extract_features(X_train)
features_train.head()

,log_retweets,log_favorites,log_replies,log_quotes,is_reply,is_quote,source_group,hour,dayofweek,log_retweets,log_favorites,log_replies,log_quotes,is_reply,is_quote,source_group_automation,source_group_instagram,source_group_linkedin,source_group_media,source_group_other,source_group_scheduler,source_group_twitter_official,source_group_unknown
0,0.0,0.0,0.0,0.0,0,1,twitter_official,13,2,0.0,0.0,0.0,0.0,0,1,False,False,False,False,False,False,True,False
1,0.0,0.0,0.0,0.0,0,1,twitter_official,13,2,0.0,0.0,0.0,0.0,0,1,False,False,False,False,False,False,True,False
2,0.0,0.0,0.0,0.0,1,0,twitter_official,13,2,0.0,0.0,0.0,0.0,1,0,False,False,False,False,False,False,True,False
3,0.0,0.0,0.0,0.0,0,0,twitter_official,13,2,0.0,0.0,0.0,0.0,0,0,False,False,False,False,False,False,True,False
4,0.0,0.0,0.0,0.0,1,0,twitter_official,13,2,0.0,0.0,0.0,0.0,1,0,False,False,False,False,False,False,True,False


In [54]:
list(features_train.columns)

['log_retweets',
 'log_favorites',
 'log_replies',
 'log_quotes',
 'is_reply',
 'is_quote',
 'source_group',
 'hour',
 'dayofweek',
 'log_retweets',
 'log_favorites',
 'log_replies',
 'log_quotes',
 'is_reply',
 'is_quote',
 'source_group_automation',
 'source_group_instagram',
 'source_group_linkedin',
 'source_group_media',
 'source_group_other',
 'source_group_scheduler',
 'source_group_twitter_official',
 'source_group_unknown']

In [64]:
features_train['log_retweets'].value_counts()

ValueError: Grouper for 'log_retweets' not 1-dimensional